In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries loaded successfully.")

Libraries loaded successfully.


Đọc dataset forecasting.

In [2]:
df = pd.read_csv("../data/forecast_dataset.csv")

# Chuyển Month sang kiểu ngày tháng
df["Month"] = pd.to_datetime(df["Month"])

print("Dataset shape:", df.shape)
print("Number of materials:", df["MaterialKey"].nunique())
print("Min month:", df["Month"].min())
print("Max month:", df["Month"].max())

df.head()

Dataset shape: (1145, 12)
Number of materials: 35
Min month: 2022-07-01 00:00:00
Max month: 2025-05-01 00:00:00


,MaterialKey,ProductID,ProductNumber,MaterialName,DSSEligibilityTier,Month,RequirementQty_PU,UnitCode,WorkOrderCount,IsFilledZero,IsTrainPeriod,IsTestPeriod
0,4,317,CA-5965,LL Crankarm,B,2024-06-01,2048.0,EA,30,0,1,0
1,4,317,CA-5965,LL Crankarm,B,2024-07-01,2976.0,EA,32,0,1,0
2,4,317,CA-5965,LL Crankarm,B,2024-08-01,3170.0,EA,32,0,1,0
3,4,317,CA-5965,LL Crankarm,B,2024-09-01,1770.0,EA,30,0,1,0
4,4,317,CA-5965,LL Crankarm,B,2024-10-01,2644.0,EA,32,0,1,0


Kiểm tra dataset trước khi EDA/model.

In [3]:
# ===== CELL 3: DATASET VALIDATION =====

print("=== 1. BASIC CHECK ===")
print("Rows:", len(df))
print("Materials:", df["MaterialKey"].nunique())

print("\n=== 2. TIER DISTRIBUTION ===")
print(
    df[["MaterialKey", "DSSEligibilityTier"]]
    .drop_duplicates()["DSSEligibilityTier"]
    .value_counts()
    .sort_index()
)

print("\n=== 3. DUPLICATE MATERIAL-MONTH ===")
duplicate_count = df.duplicated(
    subset=["MaterialKey", "Month"]
).sum()
print("Duplicate rows:", duplicate_count)

print("\n=== 4. TRAIN / TEST ROWS ===")
print(
    df.groupby(["IsTrainPeriod", "IsTestPeriod"])
      .size()
)

print("\n=== 5. OBSERVED MONTHS PER MATERIAL ===")
month_count = (
    df.groupby(
        ["MaterialKey", "ProductNumber",
         "MaterialName", "DSSEligibilityTier"]
    )
    .size()
    .reset_index(name="ObservedMonths")
)

display(month_count)

=== 1. BASIC CHECK ===
Rows: 1145
Materials: 35

=== 2. TIER DISTRIBUTION ===
DSSEligibilityTier
A    28
B     7
Name: count, dtype: int64

=== 3. DUPLICATE MATERIAL-MONTH ===
Duplicate rows: 0

=== 4. TRAIN / TEST ROWS ===
IsTrainPeriod  IsTestPeriod
0              1               204
1              0               941
dtype: int64

=== 5. OBSERVED MONTHS PER MATERIAL ===


,MaterialKey,ProductNumber,MaterialName,DSSEligibilityTier,ObservedMonths
0,4,CA-5965,LL Crankarm,B,12
1,5,CA-6738,ML Crankarm,A,35
2,6,CA-7457,HL Crankarm,B,35
3,9,CR-7833,Chainring,A,35
4,11,DC-8732,Decal 1,A,35
5,12,DC-9824,Decal 2,A,35
6,13,FH-2981,Freewheel,A,35
7,23,FC-3982,Front Derailleur Cage,A,35
8,24,FL-2301,Front Derailleur Linkage,A,35
9,25,GP-0982,Guide Pulley,A,35


kiểm tra chất lượng target

In [4]:
# ===== CELL 4: DEMAND QUALITY CHECK =====

print("=== DEMAND QUALITY ===")

print("NULL RequirementQty_PU:",
      df["RequirementQty_PU"].isna().sum())

print("Negative RequirementQty_PU:",
      (df["RequirementQty_PU"] < 0).sum())

print("Zero RequirementQty_PU:",
      (df["RequirementQty_PU"] == 0).sum())

print("Filled-zero rows:",
      (df["IsFilledZero"] == 1).sum())


print("\n=== ZERO DEMAND BY MATERIAL ===")

zero_summary = (
    df.groupby(
        ["MaterialKey", "ProductNumber",
         "MaterialName", "DSSEligibilityTier"]
    )
    .agg(
        ObservedMonths=("Month", "count"),
        ZeroMonths=("RequirementQty_PU",
                    lambda x: (x == 0).sum()),
        FilledZeroMonths=("IsFilledZero", "sum"),
        MinDemand=("RequirementQty_PU", "min"),
        MaxDemand=("RequirementQty_PU", "max"),
        AvgDemand=("RequirementQty_PU", "mean")
    )
    .reset_index()
)

display(
    zero_summary[
        zero_summary["ZeroMonths"] > 0
    ]
)

=== DEMAND QUALITY ===
NULL RequirementQty_PU: 0
Negative RequirementQty_PU: 0
Zero RequirementQty_PU: 14
Filled-zero rows: 14

=== ZERO DEMAND BY MATERIAL ===


,MaterialKey,ProductNumber,MaterialName,DSSEligibilityTier,ObservedMonths,ZeroMonths,FilledZeroMonths,MinDemand,MaxDemand,AvgDemand
2,6,CA-7457,HL Crankarm,B,35,11,11,0.0,2662.0,817.657143
20,167,RM-M464,LL Mountain Rim,B,24,1,1,0.0,1508.0,525.541667
22,169,RM-M823,HL Mountain Rim,B,23,1,1,0.0,1850.0,464.521739
24,171,RM-R600,ML Road Rim,B,35,1,1,0.0,2350.0,806.000000


EDA thống kê demand

In [5]:
# ===== CELL 5: DEMAND EDA SUMMARY =====

eda_summary = (
    df.groupby(
        ["MaterialKey", "ProductNumber",
         "MaterialName", "DSSEligibilityTier"]
    )
    .agg(
        ObservedMonths=("Month", "count"),
        MeanDemand=("RequirementQty_PU", "mean"),
        MedianDemand=("RequirementQty_PU", "median"),
        StdDemand=("RequirementQty_PU", "std"),
        MinDemand=("RequirementQty_PU", "min"),
        MaxDemand=("RequirementQty_PU", "max"),
        ZeroMonths=("RequirementQty_PU",
                    lambda x: (x == 0).sum())
    )
    .reset_index()
)

# Coefficient of Variation
eda_summary["CV"] = (
    eda_summary["StdDemand"] /
    eda_summary["MeanDemand"]
)

# Tỷ lệ tháng demand = 0
eda_summary["ZeroRate"] = (
    eda_summary["ZeroMonths"] /
    eda_summary["ObservedMonths"]
)

display(
    eda_summary.round({
        "MeanDemand": 2,
        "MedianDemand": 2,
        "StdDemand": 2,
        "MinDemand": 2,
        "MaxDemand": 2,
        "CV": 3,
        "ZeroRate": 3
    })
)

,MaterialKey,ProductNumber,MaterialName,DSSEligibilityTier,ObservedMonths,MeanDemand,MedianDemand,StdDemand,MinDemand,MaxDemand,ZeroMonths,CV,ZeroRate
0,4,CA-5965,LL Crankarm,B,12,2428.33,2642.00,953.95,422.00,4048.00,0,0.393,0.000
1,5,CA-6738,ML Crankarm,A,35,3535.94,3512.00,2204.84,40.00,9266.00,0,0.624,0.000
2,6,CA-7457,HL Crankarm,B,35,817.66,824.00,755.28,0.00,2662.00,11,0.924,0.314
3,9,CR-7833,Chainring,A,35,648.27,623.75,366.51,46.25,1611.25,0,0.565,0.000
4,11,DC-8732,Decal 1,A,35,6664.91,6264.00,3864.72,370.00,15180.00,0,0.580,0.000
5,12,DC-9824,Decal 2,A,35,3332.46,3132.00,1932.36,185.00,7590.00,0,0.580,0.000
6,13,FH-2981,Freewheel,A,35,2593.09,2495.00,1466.06,185.00,6445.00,0,0.565,0.000
7,23,FC-3982,Front Derailleur Cage,A,35,2584.69,2498.00,1454.49,185.00,6397.00,0,0.563,0.000
8,24,FL-2301,Front Derailleur Linkage,A,35,2584.69,2498.00,1454.49,185.00,6397.00,0,0.563,0.000
9,25,GP-0982,Guide Pulley,A,35,2571.54,2463.00,1437.17,185.00,6305.00,0,0.559,0.000


Kiểm tra Train/Test availability

In [6]:
# ===== CELL 6: TRAIN / TEST AVAILABILITY BY MATERIAL =====

split_summary = (
    df.groupby(
        ["MaterialKey", "ProductNumber",
         "MaterialName", "DSSEligibilityTier"]
    )
    .agg(
        TrainObserved=(
            "IsTrainPeriod",
            lambda x: (x == 1).sum()
        ),
        TestObserved=(
            "IsTestPeriod",
            lambda x: (x == 1).sum()
        )
    )
    .reset_index()
)

# Đánh dấu khả năng đánh giá trên test set chuẩn 6 tháng
split_summary["HasFullTest6Months"] = (
    split_summary["TestObserved"] == 6
)

display(split_summary)

,MaterialKey,ProductNumber,MaterialName,DSSEligibilityTier,TrainObserved,TestObserved,HasFullTest6Months
0,4,CA-5965,LL Crankarm,B,6,6,True
1,5,CA-6738,ML Crankarm,A,29,6,True
2,6,CA-7457,HL Crankarm,B,29,6,True
3,9,CR-7833,Chainring,A,29,6,True
4,11,DC-8732,Decal 1,A,29,6,True
5,12,DC-9824,Decal 2,A,29,6,True
6,13,FH-2981,Freewheel,A,29,6,True
7,23,FC-3982,Front Derailleur Cage,A,29,6,True
8,24,FL-2301,Front Derailleur Linkage,A,29,6,True
9,25,GP-0982,Guide Pulley,A,29,6,True


Tạo hàm MAE, RMSE, Bias

In [7]:
# ===== CELL 7: SHARED EVALUATION METRICS =====

def mae(actual, forecast):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)

    return np.mean(np.abs(forecast - actual))


def rmse(actual, forecast):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)

    return np.sqrt(np.mean((forecast - actual) ** 2))


def bias(actual, forecast):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)

    # Quy ước:
    # Bias > 0  -> Overforecast
    # Bias < 0  -> Underforecast
    return np.mean(forecast - actual)


print("Metric functions created successfully.")

Metric functions created successfully.


Test thử metric functions

In [8]:
# ===== CELL 8: TEST METRIC FUNCTIONS =====

actual_test = [100, 120, 140]
forecast_test = [110, 100, 150]

print("MAE :", mae(actual_test, forecast_test))
print("RMSE:", rmse(actual_test, forecast_test))
print("Bias:", bias(actual_test, forecast_test))

MAE : 13.333333333333334
RMSE: 14.142135623730951
Bias: 0.0


Naive Forecast

In [9]:
# ===== CELL 9: NAIVE FORECAST =====

naive_predictions = []
naive_metrics = []

for material_key, group in df.groupby("MaterialKey"):

    group = group.sort_values("Month")

    material_info = group.iloc[0]

    train = group[group["IsTrainPeriod"] == 1].copy()
    test = group[group["IsTestPeriod"] == 1].copy()

    # Không có train data
    if len(train) == 0:
        naive_metrics.append({
            "MaterialKey": material_key,
            "Model": "Naive",
            "MAE": np.nan,
            "RMSE": np.nan,
            "Bias": np.nan,
            "Status": "NO_TRAIN_DATA"
        })
        continue

    # Naive = giá trị cuối cùng quan sát được trong train
    last_train_value = train.iloc[-1]["RequirementQty_PU"]

    # Không đủ 6 tháng actual test
    if len(test) != 6:
        naive_metrics.append({
            "MaterialKey": material_key,
            "Model": "Naive",
            "MAE": np.nan,
            "RMSE": np.nan,
            "Bias": np.nan,
            "Status": f"INSUFFICIENT_TEST_DATA_{len(test)}_OF_6"
        })
        continue

    # Forecast 6 tháng test
    forecast_values = np.repeat(last_train_value, len(test))

    actual_values = test["RequirementQty_PU"].to_numpy()

    # Lưu prediction từng tháng
    for month, actual, forecast in zip(
        test["Month"],
        actual_values,
        forecast_values
    ):
        naive_predictions.append({
            "MaterialKey": material_key,
            "ForecastMonth": month,
            "Model": "Naive",
            "ActualQty": actual,
            "ForecastQty": forecast,
            "Error": forecast - actual
        })

    # Lưu metrics từng material
    naive_metrics.append({
        "MaterialKey": material_key,
        "Model": "Naive",
        "MAE": mae(actual_values, forecast_values),
        "RMSE": rmse(actual_values, forecast_values),
        "Bias": bias(actual_values, forecast_values),
        "Status": "SUCCESS"
    })


naive_predictions = pd.DataFrame(naive_predictions)
naive_metrics = pd.DataFrame(naive_metrics)

print("Prediction rows:", len(naive_predictions))
print("\nStatus:")
print(naive_metrics["Status"].value_counts())

display(naive_metrics)

Prediction rows: 204

Status:
Status
SUCCESS                          34
INSUFFICIENT_TEST_DATA_0_OF_6     1
Name: count, dtype: int64


,MaterialKey,Model,MAE,RMSE,Bias,Status
0,4,Naive,1132.333333,1401.328655,798.333333,SUCCESS
1,5,Naive,1473.666667,1865.071759,514.333333,SUCCESS
2,6,Naive,586.000000,698.018147,353.333333,SUCCESS
3,9,Naive,380.500000,488.390959,208.250000,SUCCESS
4,11,Naive,3755.000000,4904.430582,2269.000000,SUCCESS
5,12,Naive,1877.500000,2452.215291,1134.500000,SUCCESS
6,13,Naive,1522.000000,1953.563837,833.000000,SUCCESS
7,23,Naive,1501.833333,1931.493248,818.500000,SUCCESS
8,24,Naive,1501.833333,1931.493248,818.500000,SUCCESS
9,25,Naive,1480.333333,1907.516536,816.000000,SUCCESS


Kiểm tra kết quả Naive

In [10]:
# ===== CELL 10: VALIDATE NAIVE RESULTS =====

print("=== NAIVE PREDICTION CHECK ===")

print("Prediction rows:", len(naive_predictions))
print("Materials with predictions:",
      naive_predictions["MaterialKey"].nunique())

print("Duplicate Material-Month:",
      naive_predictions.duplicated(
          subset=["MaterialKey", "ForecastMonth", "Model"]
      ).sum())

print("NULL Actual:",
      naive_predictions["ActualQty"].isna().sum())

print("NULL Forecast:",
      naive_predictions["ForecastQty"].isna().sum())


print("\n=== NAIVE METRIC CHECK ===")

print("Metric rows:", len(naive_metrics))

print("SUCCESS:",
      (naive_metrics["Status"] == "SUCCESS").sum())

print("Non-success:",
      (naive_metrics["Status"] != "SUCCESS").sum())


print("\n=== NON-SUCCESS MATERIAL ===")

display(
    naive_metrics[
        naive_metrics["Status"] != "SUCCESS"
    ]
)


print("\n=== SAMPLE PREDICTIONS ===")

display(
    naive_predictions.head(12)
)

=== NAIVE PREDICTION CHECK ===
Prediction rows: 204
Materials with predictions: 34
Duplicate Material-Month: 0
NULL Actual: 0
NULL Forecast: 0

=== NAIVE METRIC CHECK ===
Metric rows: 35
SUCCESS: 34
Non-success: 1

=== NON-SUCCESS MATERIAL ===


,MaterialKey,Model,MAE,RMSE,Bias,Status
22,169,Naive,NaN,NaN,NaN,INSUFFICIENT_TEST_DATA_0_OF_6



=== SAMPLE PREDICTIONS ===


,MaterialKey,ForecastMonth,Model,ActualQty,ForecastQty,Error
0,4,2024-12-01,Naive,1472.0,3046.0,1574.0
1,4,2025-01-01,Naive,4048.0,3046.0,-1002.0
2,4,2025-02-01,Naive,422.0,3046.0,2624.0
3,4,2025-03-01,Naive,1930.0,3046.0,1116.0
4,4,2025-04-01,Naive,2640.0,3046.0,406.0
5,4,2025-05-01,Naive,2974.0,3046.0,72.0
6,5,2024-12-01,Naive,2790.0,4356.0,1566.0
7,5,2025-01-01,Naive,6722.0,4356.0,-2366.0
8,5,2025-02-01,Naive,934.0,4356.0,3422.0
9,5,2025-03-01,Naive,3380.0,4356.0,976.0


Moving Average Forecast

In [11]:
# ===== CELL 11: MOVING AVERAGE FORECAST =====

MA_WINDOWS = [3, 6]

ma_predictions = []
ma_metrics = []

for material_key, group in df.groupby("MaterialKey"):

    group = group.sort_values("Month")

    train = group[group["IsTrainPeriod"] == 1].copy()
    test = group[group["IsTestPeriod"] == 1].copy()

    for window in MA_WINDOWS:

        model_name = f"MA_{window}"

        # Không đủ dữ liệu train cho window
        if len(train) < window:
            ma_metrics.append({
                "MaterialKey": material_key,
                "Model": model_name,
                "MAE": np.nan,
                "RMSE": np.nan,
                "Bias": np.nan,
                "Status": f"INSUFFICIENT_TRAIN_DATA_{len(train)}"
            })
            continue

        # Không đủ 6 tháng actual test
        if len(test) != 6:
            ma_metrics.append({
                "MaterialKey": material_key,
                "Model": model_name,
                "MAE": np.nan,
                "RMSE": np.nan,
                "Bias": np.nan,
                "Status": f"INSUFFICIENT_TEST_DATA_{len(test)}_OF_6"
            })
            continue

        # Lịch sử chỉ bắt đầu từ TRAIN
        history = (
            train["RequirementQty_PU"]
            .astype(float)
            .tolist()
        )

        forecast_values = []

        # Recursive multi-step forecast
        for _ in range(len(test)):

            forecast = np.mean(history[-window:])

            forecast_values.append(forecast)

            # Chỉ thêm forecast, KHÔNG thêm actual test
            history.append(forecast)

        forecast_values = np.array(forecast_values)

        actual_values = (
            test["RequirementQty_PU"]
            .astype(float)
            .to_numpy()
        )

        # Prediction từng tháng
        for month, actual, forecast in zip(
            test["Month"],
            actual_values,
            forecast_values
        ):
            ma_predictions.append({
                "MaterialKey": material_key,
                "ForecastMonth": month,
                "Model": model_name,
                "ActualQty": actual,
                "ForecastQty": forecast,
                "Error": forecast - actual
            })

        # Metrics
        ma_metrics.append({
            "MaterialKey": material_key,
            "Model": model_name,
            "MAE": mae(actual_values, forecast_values),
            "RMSE": rmse(actual_values, forecast_values),
            "Bias": bias(actual_values, forecast_values),
            "Status": "SUCCESS"
        })


ma_predictions = pd.DataFrame(ma_predictions)
ma_metrics = pd.DataFrame(ma_metrics)


print("Prediction rows:", len(ma_predictions))

print("\nStatus:")
print(
    ma_metrics.groupby(["Model", "Status"])
              .size()
)

display(ma_metrics)

Prediction rows: 408

Status:
Model  Status                       
MA_3   INSUFFICIENT_TEST_DATA_0_OF_6     1
       SUCCESS                          34
MA_6   INSUFFICIENT_TEST_DATA_0_OF_6     1
       SUCCESS                          34
dtype: int64


,MaterialKey,Model,MAE,RMSE,Bias,Status
0,4,MA_3,955.276635,1211.961133,425.219936,SUCCESS
1,4,MA_6,947.109632,1195.869505,412.320152,SUCCESS
2,5,MA_3,1422.760402,1778.940154,239.379515,SUCCESS
3,5,MA_6,1417.917424,1767.354336,206.608296,SUCCESS
4,6,MA_3,498.331962,611.713516,123.997257,SUCCESS
...,...,...,...,...,...,...
65,180,MA_6,1364.045214,1711.052392,332.153089,SUCCESS
66,181,MA_3,1382.851623,1744.583306,394.387060,SUCCESS
67,181,MA_6,1373.110482,1720.565850,354.798983,SUCCESS
68,182,MA_3,1382.851623,1744.583306,394.387060,SUCCESS


So sánh Naive vs MA(3) vs MA(6)

In [12]:
# ===== CELL 12: COMPARE NAIVE VS MA(3) VS MA(6) =====

baseline_metrics = pd.concat(
    [naive_metrics, ma_metrics],
    ignore_index=True
)

# Chỉ những model đánh giá thành công
successful_metrics = baseline_metrics[
    baseline_metrics["Status"] == "SUCCESS"
].copy()

# Xếp hạng model theo MAE trong từng material
successful_metrics["MAE_Rank"] = (
    successful_metrics
    .groupby("MaterialKey")["MAE"]
    .rank(method="min")
)

# Model tốt nhất hiện tại theo MAE
best_baseline = (
    successful_metrics
    .sort_values(
        ["MaterialKey", "MAE", "RMSE", "Model"]
    )
    .groupby("MaterialKey")
    .first()
    .reset_index()
)

print("=== TOTAL METRIC ROWS ===")
print(len(baseline_metrics))

print("\n=== SUCCESSFUL METRIC ROWS ===")
print(len(successful_metrics))

print("\n=== BEST BASELINE COUNT ===")
print(
    best_baseline["Model"]
    .value_counts()
)

print("\n=== BEST BASELINE BY MATERIAL ===")

display(
    best_baseline[
        [
            "MaterialKey",
            "Model",
            "MAE",
            "RMSE",
            "Bias"
        ]
    ]
)

=== TOTAL METRIC ROWS ===
105

=== SUCCESSFUL METRIC ROWS ===
102

=== BEST BASELINE COUNT ===
Model
MA_6     26
MA_3      7
Naive     1
Name: count, dtype: int64

=== BEST BASELINE BY MATERIAL ===


,MaterialKey,Model,MAE,RMSE,Bias
0,4,MA_6,947.109632,1195.869505,412.320152
1,5,MA_6,1417.917424,1767.354336,206.608296
2,6,MA_6,487.850430,595.984167,123.549861
3,9,MA_6,352.115366,441.122183,92.809789
4,11,MA_3,3556.797439,4470.025592,1350.009145
5,12,MA_3,1778.398720,2235.012796,675.004572
6,13,MA_6,1408.461463,1764.488733,371.239155
7,23,MA_6,1390.903796,1746.181465,360.513642
8,24,MA_6,1390.903796,1746.181465,360.513642
9,25,MA_6,1373.110482,1720.565850,354.798983


Xuất deliverables G06

In [13]:
# ===== CELL 13: EXPORT G06 RESULTS =====

# 1. Gộp predictions của Naive + Moving Average
baseline_predictions = pd.concat(
    [naive_predictions, ma_predictions],
    ignore_index=True
)

# Sắp xếp cho dễ đọc
baseline_predictions = baseline_predictions.sort_values(
    ["MaterialKey", "Model", "ForecastMonth"]
).reset_index(drop=True)

baseline_metrics = baseline_metrics.sort_values(
    ["MaterialKey", "Model"]
).reset_index(drop=True)


# 2. Xuất dataset đầu vào
df.to_csv(
    "../output/G06_forecast_dataset.csv",
    index=False
)

# 3. Xuất prediction từng tháng
baseline_predictions.to_csv(
    "../output/G06_baseline_predictions.csv",
    index=False
)

# 4. Xuất metrics
baseline_metrics.to_csv(
    "../output/G06_baseline_metrics.csv",
    index=False
)

# 5. Xuất baseline tốt nhất hiện tại
best_baseline.to_csv(
    "../output/G06_best_baseline_by_material.csv",
    index=False
)


print("=== EXPORT COMPLETED ===")
print("Dataset rows:", len(df))
print("Prediction rows:", len(baseline_predictions))
print("Metric rows:", len(baseline_metrics))
print("Best-baseline rows:", len(best_baseline))

=== EXPORT COMPLETED ===
Dataset rows: 1145
Prediction rows: 612
Metric rows: 105
Best-baseline rows: 34


Final QA / Reconciliation

In [14]:
# ===== CELL 14: FINAL G06 QA / RECONCILIATION =====

print("=== G06 FINAL QA ===")

checks = {
    "Forecast materials = 35":
        df["MaterialKey"].nunique() == 35,

    "Dataset rows = 1145":
        len(df) == 1145,

    "Baseline prediction rows = 612":
        len(baseline_predictions) == 612,

    "Metric rows = 105":
        len(baseline_metrics) == 105,

    "Successful metric rows = 102":
        (baseline_metrics["Status"] == "SUCCESS").sum() == 102,

    "No duplicate predictions":
        baseline_predictions.duplicated(
            subset=["MaterialKey", "ForecastMonth", "Model"]
        ).sum() == 0,

    "No NULL actual in evaluated predictions":
        baseline_predictions["ActualQty"].isna().sum() == 0,

    "No NULL forecast in evaluated predictions":
        baseline_predictions["ForecastQty"].isna().sum() == 0,

    "Best baseline available for 34 materials":
        best_baseline["MaterialKey"].nunique() == 34,

    "Material 169 has no successful evaluation":
        not (
            (baseline_metrics["MaterialKey"] == 169)
            & (baseline_metrics["Status"] == "SUCCESS")
        ).any()
}

for check_name, passed in checks.items():
    print(
        f"{'PASS' if passed else 'FAIL'} - {check_name}"
    )

print("\nOverall:",
      "PASS" if all(checks.values()) else "CHECK REQUIRED")

=== G06 FINAL QA ===
PASS - Forecast materials = 35
PASS - Dataset rows = 1145
PASS - Baseline prediction rows = 612
PASS - Metric rows = 105
PASS - Successful metric rows = 102
PASS - No duplicate predictions
PASS - No NULL actual in evaluated predictions
PASS - No NULL forecast in evaluated predictions
PASS - Best baseline available for 34 materials
PASS - Material 169 has no successful evaluation

Overall: PASS


## G06 – Baseline Forecasting: Naive & Moving Average

### 1. Forecasting scope

The forecasting dataset contains **35 eligible materials** within the standardized forecast window from **July 2022 to May 2025**.

Material requirements are expressed in the material's purchasing unit using `RequirementQty_PU`. This ensures that materials requiring unit conversion are forecast in a unit suitable for subsequent procurement and DSS analysis.

The forecasting window is divided chronologically into:

- **Training period:** July 2022 – November 2024
- **Test period:** December 2024 – May 2025
- **Test horizon:** 6 months

A chronological split is used instead of a random split because the objective is to evaluate forecasting performance on future periods using only information available before those periods.

### 2. Data-quality handling

The dataset was validated before forecasting.

Key validation results:

- 35 materials are included in the forecasting scope.
- No duplicate Material–Month observations were found.
- No NULL or negative values were found in existing `RequirementQty_PU` observations.
- Zero-demand observations explicitly generated by the ETL process (`IsFilledZero = 1`) are retained as valid observations.
- Missing calendar observations are **not automatically interpreted as zero demand**.

Most materials contain the complete forecasting history. Some Tier B materials contain shorter histories and are therefore evaluated only where sufficient observations are available.

### 3. Baseline models

Three baseline configurations were evaluated:

1. **Naive**
2. **Moving Average – 3 months (MA_3)**
3. **Moving Average – 6 months (MA_6)**

For the Naive model, all six test-period forecasts are based on the final observed training-period value.

For Moving Average, a **recursive multi-step forecasting strategy** is used. After forecasting the first future month, that forecast is appended to the available history and may be used to generate the following forecast. Actual test-period values are never inserted into the forecasting history.

This prevents test-data leakage and simulates the information available when forecasting multiple future periods.

### 4. Evaluation metrics

All successfully evaluated models use the same metrics:

- **MAE (Mean Absolute Error):** primary accuracy metric.
- **RMSE (Root Mean Squared Error):** gives greater weight to large forecast errors.
- **Bias:** calculated as `Forecast - Actual`.

Bias interpretation:

- Bias > 0: tendency to overforecast.
- Bias < 0: tendency to underforecast.
- Bias close to 0: forecast errors are approximately balanced.

Using a common evaluation framework allows subsequent forecasting methods to be compared consistently.

### 5. Baseline evaluation results

For each of the three baseline configurations:

- **34 materials** were successfully evaluated over the complete six-month test period.
- **1 material (MaterialKey 169)** could not be evaluated because it has no actual observations in the six-month test period.

Material 169 is retained in the results with the explicit status:

`INSUFFICIENT_TEST_DATA_0_OF_6`

No artificial zero-demand observations were created to calculate its evaluation metrics.

Across the 34 evaluable materials, the model with the lowest MAE within the current baseline family was:

- **MA_6:** 26 materials
- **MA_3:** 7 materials
- **Naive:** 1 material

Therefore, Moving Average—particularly MA_6—performs better than the Naive baseline for most materials in the current backtest.

### 6. Interpretation and next step

The best baseline identified in G06 is **not the final forecasting model used by the DSS**.

These results establish a benchmark for subsequent forecasting approaches. ETS/Holt-Winters and ARIMA models will be evaluated using the same dataset, chronological train/test split, six-month test horizon, and MAE/RMSE/Bias framework.

Final model selection will be performed only after all candidate model families have been evaluated under the common framework.

### 7. G06 output

G06 produces the following reusable outputs:

- `G06_forecast_dataset.csv`
- `G06_baseline_predictions.csv`
- `G06_baseline_metrics.csv`
- `G06_best_baseline_by_material.csv`

Final reconciliation checks confirm:

- 35 materials in forecasting scope
- 1,145 input observations
- 612 baseline prediction records
- 105 model-metric records
- 102 successful model evaluations
- no duplicate prediction records
- no NULL actual or forecast values in evaluated predictions
- explicit handling of the unevaluable Material 169

**G06 final QA status: PASS.**